<a href="https://colab.research.google.com/github/zerogradlabs/zerograd-python/blob/main/examples/no-target.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Without a target: anomalies, gaps, synthetic rows and segments

Four tasks with no labels, with tabular foundation models and the open source SDK, locally. Each one runs the classic scikit-learn method next
to the models whenever there is something to measure, so you see what the model adds.

| task | foundation models | against |
|---|---|---|
| `zg.anomalies` | TabPFN v2, TabICLv2, FoMo-0D | Isolation Forest, LOF, kNN |
| `zg.impute` | TabPFN v2, TabICLv2 | mean/mode, KNN, iterative imputation |
| `zg.synthesize` | TabPFN v2, TabICLv2 | sampling each column on its own |
| `zg.cluster` | ZEUS | k-means, Gaussian mixture |

Every model here allows commercial use. TabPFN v2 asks you to show "Built with PriorLabs-TabPFN".

In [ ]:
!pip install -q zerograd

In [ ]:
import numpy as np
import pandas as pd

import zerograd
from zerograd import ZeroGrad

zg = ZeroGrad()
zerograd.list_tasks()     # every task and which models run it

## The data

Card transactions: the amount follows the merchant category and the hour, international ones are rarer, and 2% are fraud that breaks the
pattern (a big amount at a grocery store at 4 am).

In [ ]:
rng = np.random.default_rng(0)
n = 1500
category = rng.choice(["grocery", "fuel", "travel", "electronics"], n, p=[0.45, 0.25, 0.15, 0.15])
base = pd.Series(category).map({"grocery": 40, "fuel": 55, "travel": 400, "electronics": 250}).to_numpy()
hour = np.clip(rng.normal(14, 4, n), 0, 23).round()
tx = pd.DataFrame({
    "category": category,
    "hour": hour,
    "amount": (base * rng.lognormal(0, 0.35, n)).round(2),
    "international": rng.random(n) < np.where(category == "travel", 0.6, 0.05),
    "fraud": 0,
})
bad = rng.choice(n, 30, replace=False)
tx.loc[bad, "amount"] = (tx.loc[bad, "amount"] * rng.uniform(8, 15, 30)).round(2)
tx.loc[bad, "hour"] = rng.integers(1, 5, 30)
tx.loc[bad, "fraud"] = 1
tx.head()

## 1. Anomaly detection

With known labels (`labels="fraud"`), every method is scored by how well it ranks the fraud first and the best one wins. Without labels, the
first foundation model scores the rows. The labels are never read as a feature.

In [ ]:
found = zg.anomalies(tx, labels="fraud")
found.evaluation

Without labels, the first foundation model scores every row. TabPFN v2 and TabICLv2 also say which columns make each row unusual
(from the model's own per-column factors), in `why`:

In [ ]:
unlabeled = zg.anomalies(tx.drop(columns="fraud"), models=["tabpfn-v2"])
unlabeled.top(10)

In [ ]:
unlabeled.scores.loc[bad, "is_anomaly"].mean()    # share of the planted fraud it flagged, without ever seeing a label

Score new rows against a clean reference instead, for example today's transactions against last month's:

In [ ]:
today = zg.anomalies(tx.drop(columns="fraud").iloc[:200], reference=tx[tx.fraud == 0].drop(columns="fraud").iloc[200:])
today.anomalies.head()

## 2. Imputation

Some cells go missing. `impute` first hides known cells and scores every method on them, then the champion fills the real gaps from the rest
of each row.

In [ ]:
holes = tx.drop(columns="fraud").copy()
holes.loc[rng.choice(n, 150, replace=False), "amount"] = np.nan
holes.loc[rng.choice(n, 100, replace=False), "category"] = None
filled = zg.impute(holes)
filled.evaluation

In [ ]:
filled.data[filled.filled.any(axis=1)].head()

## 3. Synthetic data

New rows with the same columns and relations. A classifier then tries to tell them from the real ones: a detection AUC near 0.5 means it
can't. `exact_copies` is the share of synthetic rows that copy a real row.

In [ ]:
fake = zg.synthesize(tx.drop(columns="fraud"), n=1000)
fake.evaluation

In [ ]:
fake.data.groupby("category")["amount"].median().round(1), tx.groupby("category")["amount"].median().round(1)

## 4. Clustering

The wine dataset that ships with scikit-learn: 178 wines, 13 measurements, three cultivars. With `labels=` the known groups score each method
(adjusted Rand index); without, ZEUS picks the segments and `k` is chosen for you, or pass `k=3`.

In [ ]:
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True).frame
groups = zg.cluster(wine, labels="target")
groups.evaluation

In [ ]:
groups.profiles

## Next

* The [evaluation notebook](https://github.com/zerogradlabs/zerograd-python/blob/main/examples/evaluation.ipynb): cross-validation by group and time, thresholds, importance and scikit-learn.
* Docs: [zerograd.dev/docs#no-target](https://zerograd.dev/docs#no-target).